In [1]:
import pandas as pd

anime = pd.read_csv('anime.csv', low_memory=False)
rating = pd.read_csv('rating_complete.csv', low_memory=False)
synopsis = pd.read_csv('anime_with_synopsis.csv', low_memory=False)
animelist = pd.read_csv('animelist.csv', low_memory=False)

In [2]:
anime = anime.rename(columns={'MAL_ID': 'anime_id'})
synopsis = synopsis.rename(columns={'MAL_ID': 'anime_id'})

In [3]:
anime_data = pd.merge(anime, synopsis.drop(columns=['Name', 'Score', 'Genres']), on='anime_id')
anime_data['Score']=anime_data['Score'].replace('Unknown', 0).astype(float)
len(anime_data)

16214

In [6]:
animelist['watching_status'].unique()

array([ 1,  2,  3,  4,  6,  0,  5, 33, 55])

In [7]:
animelist = animelist.drop(animelist[animelist['watching_status'].isin([0, 5, 33, 55])].index)

In [8]:
animelist['watching_status'].unique()

array([1, 2, 3, 4, 6])

In [9]:
# Give each status from watching_status values
status_weight_map = {
    1: 0.8, # Currently Watching
    2: 1.0, # Completed
    3: 0.3, # On Hold
    4: -0.5, # Dropped
    6: 0.2 # Plan to watch
}
anime_status = animelist[['anime_id','watching_status']].copy()
anime_status['status_score']=anime_status['watching_status'].map(status_weight_map)

In [10]:
anime_status['status_score'].unique()

array([ 0.8,  1. ,  0.3, -0.5,  0.2])

In [11]:
anime_rating = pd.DataFrame(rating.groupby('anime_id').agg(rating = ('rating', 'mean'), vote = ('rating', 'sum'))).reset_index()
anime_rating['rating'] = anime_rating['rating'].fillna(0)

In [ ]:
m = anime_rating['vote'].quantile(0.9)
m

np.float64(18985.4)

In [13]:
qualified_anime = anime_rating.copy().loc[anime_rating['vote'] >= m]
len(qualified_anime)

3375

In [14]:
rating = rating[rating['anime_id'].isin(qualified_anime['anime_id'])]

In [15]:
# Filter ratings to only include qualified anime BEFORE creating matrix
rating_filtered = rating[rating['anime_id'].isin(qualified_anime['anime_id'])]

# Now create the user-anime matrix with filtered data
user_anime_matrix = rating_filtered.pivot_table(
    index='user_id', 
    columns='anime_id', 
    values='rating'
).fillna(0)

print(f"Matrix shape: {user_anime_matrix.shape}")
print(f"Memory usage: ~{user_anime_matrix.memory_usage(deep=True).sum() / 1024**3:.2f} GB")

Matrix shape: (309640, 3375)
Memory usage: ~7.79 GB


In [16]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
tfidf = TfidfVectorizer(stop_words='english')
anime_data['sypnopsis'] = anime_data['sypnopsis'].fillna('')
tfidf_matrix = tfidf.fit_transform(anime_data['sypnopsis'])
cosine_sim = cosine_similarity(tfidf_matrix)

In [17]:
def get_recommendations(anime_name, cosine_sim = cosine_sim):
    # Collaborative Filtering
    anime_id = anime[anime['Name'] == anime_name]['anime_id'].values[0]
    
    users_ratings = user_anime_matrix[anime_id]
    cf_scores = user_anime_matrix.corrwith(users_ratings)
    cf_scores = pd.DataFrame(cf_scores, columns=['cf_score']).dropna()

    # Conetent Based
    anime_index = synopsis[synopsis['anime_id'] == anime_id].index[0]
    cb_scores = pd.Series(cosine_sim[anime_index], index=synopsis['anime_id'])
    cb_scores = pd.DataFrame(cb_scores, columns=['cb_score'])

    # Score for watching_status of that anime
    ws_scores = anime_status.groupby('anime_id')['status_score'].mean().reset_index()

    #Merge all score
    scores = cf_scores.merge(cb_scores, on='anime_id').fillna(0)
    scores = scores.merge(ws_scores[['anime_id','status_score']], on='anime_id').fillna(0)
    scores['final_score'] = 0.6 * scores['cf_score'] + 0.3 * scores['cb_score'] + 0.1 * scores['status_score']

    df_name = anime[['anime_id','Name']]
    df_synopsis = synopsis[['anime_id','sypnopsis']]
    result = scores.merge(df_name, on='anime_id')
    result = result.merge(df_synopsis, on='anime_id')

    return result.sort_values('final_score', ascending=False).head(10)

In [18]:
anime_id = anime[anime['Name'] == 'Naruto']['anime_id'].values[0]
users_ratings = user_anime_matrix[anime_id]
cf_scores = user_anime_matrix.corrwith(users_ratings)
cf_scores = pd.DataFrame(cf_scores, columns=['cf_score']).dropna()
anime_index = synopsis[synopsis['anime_id'] == anime_id].index[0]
cb_scores = pd.Series(cosine_sim[anime_index], index=synopsis['anime_id'])
cb_scores = pd.DataFrame(cb_scores, columns=['cb_score'])
ws_scores = anime_status.groupby('anime_id')['status_score'].mean().reset_index()
scores = cf_scores.merge(cb_scores, on='anime_id').fillna(0)
scores = scores.merge(ws_scores[['anime_id','status_score']], on='anime_id').fillna(0)
print(scores)

      anime_id  cf_score  cb_score  status_score
0            1  0.096623  0.005751      0.715153
1            5  0.043276  0.008769      0.843339
2            6  0.121123  0.010476      0.695798
3            7  0.038856  0.007631      0.560456
4           15  0.138031  0.000000      0.569598
...        ...       ...       ...           ...
3334     41619  0.050635  0.000000      0.676122
3335     41783  0.033041  0.000000      0.514848
3336     41930  0.043465  0.008742      0.610733
3337     42603  0.056559  0.003610      0.723673
3338     42984  0.029607  0.000000      0.926070

[3339 rows x 4 columns]


In [19]:
user_input = input('Enter anime name that you want to get similar rating anime: ')
get_recommendations(user_input)

Enter anime name that you want to get similar rating anime:  Shingeki no Kyojin


,anime_id,cf_score,cb_score,status_score,final_score,Name,sypnopsis
1911,16498,1.000000,1.000000,0.913011,0.991301,Shingeki no Kyojin,"Centuries ago, mankind was slaughtered to near..."
2320,25777,0.612210,0.314828,0.877483,0.549523,Shingeki no Kyojin Season 2,"For centuries, humanity has been hunted by gia..."
2943,35760,0.497184,0.305084,0.871234,0.476959,Shingeki no Kyojin Season 3,"Still threatened by the ""Titans"" that rob them..."
3179,38524,0.460379,0.276234,0.915076,0.450605,Shingeki no Kyojin Season 3 Part 2,Seeking to restore humanity's diminishing hope...
2187,22319,0.514504,0.065014,0.884128,0.416619,Tokyo Ghoul,Tokyo has become a cruel and merciless city—a ...
2448,30276,0.479410,0.020902,0.919814,0.385898,One Punch Man,The seemingly ordinary and unimpressive Saitam...
2615,31964,0.455816,0.014402,0.919939,0.369804,Boku no Hero Academia,"The appearance of ""quirks,"" newly discovered s..."
1652,11757,0.451633,0.025798,0.871890,0.365908,Sword Art Online,"In the year 2022, virtual reality has progress..."
2358,27899,0.426242,0.022649,0.890467,0.351587,Tokyo Ghoul √A,Ken Kaneki has finally come to accept the mons...
2088,20507,0.434076,0.014514,0.835706,0.348370,Noragami,"In times of need, if you look in the right pla..."


In [92]:
anime_rating.loc[anime_rating['anime_id'] == 21, 'vote'].values[0]

np.int64(12)